In [8]:

import pandas as pd

movies= pd.read_csv("../data/movies.csv")
ratings= pd.read_csv("../data/ratings.csv")

In [9]:
movies.shape

(10329, 3)

In [10]:
ratings.shape

(105339, 4)

In [11]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [13]:
print(movies.columns.tolist())
print(ratings.columns.tolist())


['movieId', 'title', 'genres']
['userId', 'movieId', 'rating', 'timestamp']


In [14]:
movies['genres']= movies['genres'].str.replace('|', " ")

In [15]:
movies.head(
)

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf= TfidfVectorizer(stop_words='english')
tfidf_matrix= tfidf.fit_transform(movies['genres'])

print(tfidf_matrix.shape)

(10329, 23)


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim= cosine_similarity(tfidf_matrix,tfidf_matrix)
print(cosine_sim)

[[1.         0.79977247 0.1589222  ... 0.2638368  0.         0.        ]
 [0.79977247 1.         0.         ... 0.         0.         0.        ]
 [0.1589222  0.         1.         ... 0.60235038 0.         0.        ]
 ...
 [0.2638368  0.         0.60235038 ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]


In [18]:
indices= pd.Series(movies.index, index=movies['title'])

In [21]:
def get_recommendation(title, n=10):
    idx= indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    movies_indices = [i[0] for i in sim_scores]
    return movies["title"].iloc[movies_indices]


In [23]:
get_recommendation("Toy Story (1995)")

1815                                          Antz (1998)
2496                                   Toy Story 2 (1999)
2967       Adventures of Rocky and Bullwinkle, The (2000)
3166                     Emperor's New Groove, The (2000)
3811                                Monsters, Inc. (2001)
6617    DuckTales: The Movie - Treasure of the Lost La...
6997                                     Wild, The (2006)
7382                               Shrek the Third (2007)
7987                       Tale of Despereaux, The (2008)
9215    Asterix and the Vikings (Astérix et les Viking...
Name: title, dtype: object

In [24]:
ratings = pd.read_csv("../data/ratings.csv")
print(ratings.head())
print(ratings.shape)
print(ratings['userId'].nunique())   # how many unique users?
print(ratings['movieId'].nunique())  # how many unique movies rated?

   userId  movieId  rating   timestamp
0       1       16     4.0  1217897793
1       1       24     1.5  1217895807
2       1       32     4.0  1217896246
3       1       47     4.0  1217896556
4       1       50     4.0  1217896523
(105339, 4)
668
10325


In [26]:
from surprise import SVD, Dataset, Reader
print("surprise imported successfully")

surprise imported successfully


In [27]:
from surprise import SVD, Reader, Dataset
from surprise.model_selection import train_test_split as surprise_split

reader = Reader(rating_scale=(0.5, 5))
data = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], reader)

train_set, test_set = surprise_split(data, test_size=0.2)
model= SVD(random_state=42)
model.fit(train_set)

print("done")

done


In [28]:
# predict rating for user 1 on movieId 1 (Toy Story)
prediction = model.predict(uid=1, iid=1)
print(prediction)
print("Predicted rating:", prediction.est)

user: 1          item: 1          r_ui = None   est = 3.70   {'was_impossible': False}
Predicted rating: 3.7043162328089143


In [29]:
# get all movies user 1 hasn't rated yet
user_1_rated = ratings[ratings['userId'] == 1]['movieId'].tolist()
unrated = movies[~movies['movieId'].isin(user_1_rated)]

# predict ratings for all unrated movies
predictions = []
for _, row in unrated.iterrows():
    pred = model.predict(uid=1, iid=row['movieId'])
    predictions.append((row['title'], pred.est))

# sort by predicted rating
predictions.sort(key=lambda x: x[1], reverse=True)
predictions[:10]

[('Ran (1985)', np.float64(4.456802728169629)),
 ('Seven Samurai (Shichinin no samurai) (1954)',
  np.float64(4.453962450870478)),
 ('Notorious (1946)', np.float64(4.416500957763852)),
 ('Paths of Glory (1957)', np.float64(4.401057726426115)),
 ('Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)',
  np.float64(4.385512745098912)),
 ('Princess Mononoke (Mononoke-hime) (1997)', np.float64(4.364709025970386)),
 ('Dark Knight, The (2008)', np.float64(4.319142671668127)),
 ('Apocalypse Now (1979)', np.float64(4.313261856303399)),
 ('Touch of Evil (1958)', np.float64(4.313121237580657)),
 ('Annie Hall (1977)', np.float64(4.30604143945957))]

In [30]:
def hybrid_recommendations(user_id, title, movies, cosine_sim, indices, model, n=10):
    # get content based scores
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))

    # get collaborative scores for each movie
    results = []
    for movie_idx, content_score in sim_scores:
        movie_id = movies.iloc[movie_idx]['movieId']
        collab_score = model.predict(uid=user_id, iid=movie_id).est
        hybrid_score = content_score + collab_score
        results.append((movie_idx, hybrid_score))

    # sort by hybrid score
    results = sorted(results, key=lambda x: x[1], reverse=True)
    results = results[1:n+1]
    movie_indices = [i[0] for i in results]
    return movies['title'].iloc[movie_indices].tolist()

# test it
print(hybrid_recommendations(1, "Toy Story (1995)", movies, cosine_sim, indices, model))

['Princess Mononoke (Mononoke-hime) (1997)', 'Nausicaä of the Valley of the Wind (Kaze no tani no Naushika) (1984)', 'Toy Story 3 (2010)', 'Monsters, Inc. (2001)', 'Toy Story 2 (1999)', 'Wallace & Gromit: A Close Shave (1995)', 'Spirited Away (Sen to Chihiro no kamikakushi) (2001)', "Howl's Moving Castle (Hauru no ugoku shiro) (2004)", 'Up (2009)', 'Lord of the Rings: The Fellowship of the Ring, The (2001)']
